<div style="background:linear-gradient(120deg,#07062e 0%,#120c5c 60%,#2a1a9e 100%);border-radius:14px;padding:26px 30px;color:#ffffff;font-family:'Segoe UI',Arial,sans-serif;"><div style="font-size:12px;letter-spacing:3px;color:#4de1ff;font-weight:600;">MICROSOFT IGNITE 2026 &nbsp;&middot;&nbsp; LAB663</div><div style="font-size:30px;font-weight:700;margin:8px 0 4px 0;color:#ffffff;">Module 2 &middot; Choosing a model</div><div style="font-size:15px;color:#cfd4ff;">Set the stage &middot; about 8 minutes</div><div style="margin-top:16px;"><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">1</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#4de1ff;color:#07062e;">2</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">3</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">4</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">5</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:transparent;color:#cfd4ff;border:1px solid #5a54c9;">6</span></div><div style="height:6px;border-radius:3px;margin-top:18px;background:linear-gradient(90deg,#3cf26b,#4de1ff,#6c4dff,#ff4fd8,#ff9f1c,#ffd23f);"></div></div>

# 2. Choosing a model: the tradeoffs

**Time:** about 8 minutes.

**The question we're starting with:** which model should I use?

Every model is a trade. A bigger model is usually smarter and also slower and pricier. A smaller one is quick and cheap, and might stumble on a hard question. There's no "best" model, only the right one for *this* job.

In this notebook we'll feel those trade-offs with our own eyes. We'll send the **same questions** to two models, then change one dial (reasoning effort) and watch what moves.

By the end we'll have a rule of thumb for choosing, and a sharp question for Module 6: *what if we didn't have to choose by hand?*


---

## 2.1 What are we comparing?

Same helpers as before. We'll also use `pandas` to line results up in a table, because comparisons are easier to read side by side.

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
import lab

# Connect with our own Azure sign-in, so there are no keys to manage.
project_client, openai_client = lab.connect()
print("Connected. Comparing:", lab.FRONTIER_MODEL, "and", lab.SMALL_MODEL)

## 2.2 What should I ask them?

We'll hand each model the product manuals *inside the prompt*, so they can answer from facts and not memory. (Our real agent will use file search for this. Here we keep it simple so we only compare the models.)

- **Easy:** one fact lookup.
- **Medium:** compare two products.
- **Hard:** juggle several needs at once and pick a winner.

**Before you run anything, predict:** which question will the small model get wrong, if any? Which will be the fastest? Write a guess in your head (or on paper). We'll check it in a minute.

In [ ]:
# Give the models the two tent manuals as context so they answer from facts, not memory.
context = (
    "Here are two product manuals.\n\n=== TrailMaster X4 Tent ===\n" + lab.manual_text(1)
    + "\n\n=== Alpine Explorer Tent ===\n" + lab.manual_text(8)
)

questions = {
    "easy": "What is the rainfly waterproof rating of the TrailMaster X4 Tent?",
    "medium": "Which tent is lighter, and by how many pounds?",
    "hard": (
        "A family of six is camping in heavy rain and wants to carry as little weight as possible. "
        "Which tent should they take? Explain the trade-off in two sentences."
    ),
}
for level, question in questions.items():
    print(f"{level:>6}: {question}")

## 2.3 How do the two models compare on speed and tokens?

The helper below sends one question to one model and times it. We keep three numbers for each answer: **seconds**, **tokens** (a token is roughly three-quarters of a word, and it's what we pay for), and the answer itself.

In [ ]:
def ask(model, question, effort=None):
    """Send one question to one model. Return the answer plus how long and how many tokens it took."""
    options = {"reasoning": {"effort": effort}} if effort else {}
    started = time.perf_counter()
    reply = openai_client.responses.create(model=model, input=f"{context}\n\nQuestion: {question}", **options)
    return {
        "seconds": round(time.perf_counter() - started, 1),
        "input_tokens": reply.usage.input_tokens,
        "output_tokens": reply.usage.output_tokens,
        "answer": reply.output_text.strip(),
    }


# Every question goes to both models, so the comparison is fair.
rows = []
for level, question in questions.items():
    for model in [lab.FRONTIER_MODEL, lab.SMALL_MODEL]:
        rows.append({"level": level, "model": model, **ask(model, question)})

table = pd.DataFrame(rows)
# Numbers first: speed and token counts side by side. The answers come next.
table[["level", "model", "seconds", "input_tokens", "output_tokens"]]

**Look at the numbers first.** Which model was faster? Did the gap change as the questions got harder? Notice the *input* tokens: they're the same for both, because we sent the same manuals. The model we choose changes the price of those tokens.

## 2.4 But are the quicker answers any good?

Numbers only tell half the story. Speed and cost mean nothing if the answer is wrong. Read each pair and ask: **would a customer be happy with this?**

In [ ]:
# Print each model's answer under the question it was answering.
for level in questions:
    print(f"\n===== {level.upper()} =====")
    for _, row in table[table["level"] == level].iterrows():
        print(f"\n[{row['model']}]  {row['seconds']}s")
        print(row["answer"])

**Think:**
- Check your prediction. Did the small model slip on the hard one, or did it hold up?
- For the *easy* question, was the bigger model worth waiting for?
- If you ran a store with thousands of questions a day, how many of them look like the easy one?

Hold that last thought. It's the whole reason Module 6 exists.

## 2.5 What will this cost me?

Tokens are only half of cost. The other half is the **price per token**, and that differs by model and region. We keep prices in `data/pricing.json`, and we've left them empty on purpose: prices change, and we never want to quote a made-up number. Fill them in from the Azure pricing page to turn the table into dollars.

In [ ]:
def dollars(row):
    # Returns None until the model's prices are filled in data/pricing.json.
    return lab.cost_of(row["model"], row["input_tokens"], row["output_tokens"])


table["dollars"] = table.apply(dollars, axis=1)

if table["dollars"].isna().all():
    print("No prices yet. Open data/pricing.json, fill in the numbers, and re-run this cell.")
    print("Until then, compare output_tokens: more tokens out means a pricier answer.")
else:
    print(table[["level", "model", "dollars"]])

table.groupby("model")[["seconds", "output_tokens"]].sum()

## 2.6 Does thinking harder pay off?

Newer models can **think harder before they answer**. We turn that dial with `reasoning effort`. More thinking can mean better answers on hard problems, and it can also mean more tokens and more waiting.

Let's turn the dial on the *hard* question with the bigger model and see what we get.

In [ ]:
# Same hard question, same model: only the amount of thinking changes.
effort_rows = []
for effort in ["low", "high"]:
    try:
        result = ask(lab.FRONTIER_MODEL, questions["hard"], effort=effort)
        effort_rows.append({"effort": effort, **{k: v for k, v in result.items() if k != "answer"}})
        print(f"\n[{effort}] {result['answer']}")
    except Exception as error:
        print(f"[{effort}] this deployment didn't accept that setting: {str(error)[:120]}")

pd.DataFrame(effort_rows)

**Think:** did "high" give a meaningfully better answer, or just a longer wait and a bigger bill? For a question this size, was the extra thinking worth it?

## 2.7 So how do I choose?

Here's the pattern we keep seeing:

| Question looks like... | Reach for... | Why |
|---|---|---|
| A quick lookup | A small, fast model | It gets it right and costs less |
| A comparison or recommendation | A stronger model | Reasoning matters here |
| Something tricky with many constraints | A strong model, maybe more reasoning effort | Mistakes are expensive |

The catch: *someone has to decide, for every question, which row it belongs to.* Doing that by hand doesn't scale. Keep that in mind.

## 2.8 What if the easy questions went to the small model? (the aha)

We've been asking "which model?" Let's test a better question with our own numbers: **which questions need the big one?**

In [ ]:
# What if only the hard question went to the big model and the rest went to the small one?
big, small = lab.FRONTIER_MODEL, lab.SMALL_MODEL

all_big = table[table["model"] == big]["output_tokens"].sum()
mixed = table[
    ((table["level"] != "hard") & (table["model"] == small))
    | ((table["level"] == "hard") & (table["model"] == big))
]["output_tokens"].sum()

print(f"Everything on {big}: {all_big} output tokens")
print(f"Easy and medium on {small}, hard on {big}: {mixed} output tokens")
print(f"About {round(100 * (1 - mixed / all_big))}% fewer output tokens, if the small model's answers held up.")

> **Aha:** "Which model is best?" was the wrong question. The right one is "**which of my questions need the big model?**" The answer is usually "fewer than you think", and that gap is where the savings live.

We also took away:

- **No best model, only a best fit.** Size, speed, cost and capability pull against each other.
- We should **always check the answer**, not just the speed and price.
- **Reasoning effort** is a second dial that trades time and tokens for care.
- Most real traffic is easy. Paying frontier prices for easy questions is the hidden cost.

**Next:** `03-multi-model-vs-multimodal.ipynb`. What if one task needs several models, and what if the customer sends a photo?